# Complete regional SCHISM case study

**Learning goals:** Assemble the Australian regional case with real mesh, vertical grid, ERA5, HYCOM, and tidal inputs, then generate its workspace.

**Prerequisites:** Lessons 1–5; shared fixture data. A SCHISM executable and MPI are only needed for the optional runtime cell.

**Execution contract:** This lesson is **configuration-only**. Documentation rendering never executes SCHISM, downloads data, or requires MPI/Docker.

## Checkpoint

By the end of this lesson, record what was configured and which steps still require a model runtime.

Previous: [journey_05_schism_boundaries](../journey_05_schism_boundaries/)

Next: [journey_07_schism_execution](../journey_07_schism_execution/)


## Why this matters: SCHISM data preparation

**Without Rompy:** preparing HYCOM boundary conditions can mean downloading a large global dataset, selecting the run period and region, interpolating onto open-boundary nodes, extracting the required variables, and writing `elev2D.th.nc` or other SCHISM files. ERA5 and tidal inputs require similarly separate preparation steps.

**With Rompy:** source objects, grid metadata, time ranges, filters, and boundary mappings are assembled into `SCHISMConfig`. Workspace generation carries out the configured cropping, interpolation, boundary extraction, and SCHISM-format conversion. The modeller still chooses appropriate datasets, variables, coordinates, numerical settings, and scientific validation checks.

The following cells show the source fields, model domain, and generated artefacts so this automation remains inspectable.


In [ ]:
import sys
from pathlib import Path

root = next(path for path in [Path.cwd(), *Path.cwd().parents]
             if (path / "scripts" / "schism_case_data.py").is_file())
sys.path.insert(0, str(root))
from scripts.schism_case_data import ensure_schism_data

case = ensure_schism_data()
print("Fixture directory:", case)


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from rompy.core.data import DataBlob
from rompy.core.filters import Filter
from rompy.core.source import SourceFile
from rompy.core.time import TimeRange
from rompy.model import ModelRun
from rompy_schism import SCHISMGrid, SCHISMConfig
from rompy_schism.data import (
    SCHISMData, SCHISMDataSflux, SfluxAir, SCHISMDataBoundary,
)
from rompy_schism.boundary_conditions import create_hybrid_boundary_config
from rompy_schism.namelists import NML
from rompy_schism.namelists.param import Param, Core

# Build the complete case from real shared fixtures.
grid = SCHISMGrid(
    hgrid=DataBlob(source=case / "hgrid.gr3"),
    vgrid=DataBlob(source=case / "vgrid.in"),
    drag=1,
)
atmos = SCHISMDataSflux(
    air_1=SfluxAir(
        source=SourceFile(uri=case / "era5.nc"),
        filter=Filter(sort={"coords": ["latitude"]}),
        uwind_name="u10",
        vwind_name="v10",
        buffer=2,
    )
)
boundaries = create_hybrid_boundary_config(
    constituents=["M2", "S2", "N2"],
    tidal_database=case / "tides",
    tidal_model="OCEANUM-atlas",
    elev_source=SCHISMDataBoundary(
        id="elev2D",
        source=SourceFile(uri=case / "hycom.nc"),
        variables=["surf_el"],
        coords={"t": "time", "y": "ylat", "x": "xlon"},
    ),
)
config = SCHISMConfig(
    grid=grid,
    data=SCHISMData(atmos=atmos, boundary_conditions=boundaries),
    nml=NML(param=Param(core=Core(rnday=0.1))),
)

with TemporaryDirectory() as output:
    run = ModelRun(
        run_id="regional_case",
        output_dir=output,
        period={"start": "2023-01-01", "end": "2023-01-02", "interval": "1h"},
        config=config,
    )
    workspace = Path(run.generate())
    print("Generated complete workspace:", workspace)
    print("Generated files:", len(list(workspace.rglob("*"))))
    from scripts.notebook_helpers import print_workspace_manifest
    generated_files = print_workspace_manifest(workspace)
    # Read back the generated model inputs while the temporary workspace exists.
    from scripts.schism_case_data import assert_netcdf_contract
    sflux = next(workspace.glob("sflux/air_*.nc"))
    assert_netcdf_contract(sflux, variables=("u10", "v10", "prmsl"), dimensions=("time", "nx_grid"))
    assert_netcdf_contract(workspace / "elev2D.th.nc", variables=("time_series",), dimensions=("time", "nOpenBndNodes"))
    print("Sflux read-back:", sflux.name)
    print("Elevation boundary read-back:", (workspace / "elev2D.th.nc").name)
    print("bctides preview:", (workspace / "bctides.in").read_text().splitlines()[:4])


## Verify the generated SCHISM workspace

The complete case produces model-ready files without launching a SCHISM binary. The manifest below makes the hand-off explicit: `sflux/*` contains atmospheric forcing, `bctides.in` contains tidal boundary metadata, and `elev2D.th.nc` contains HYCOM-derived boundary values.


In [ ]:
# The generation cell kept the manifest before the temporary workspace was removed.
expected = ["bctides.in", "elev2D.th.nc"]
assert all(any(name.endswith(item) for name in generated_files) for item in expected)
assert any(name.startswith("sflux/") for name in generated_files)
print("Verified real-data transformations:", expected + ["sflux/*"])


## Forcing inventory and honest coupling boundary

This verification section separates prepared inputs from scientific validation. The key assumption is that each source variable, unit, coordinate, and time range has been checked for the intended experiment.

This case prepares atmospheric Sflux, tidal metadata, and HYCOM elevation input. The same HYCOM fixture also contains 3-D ocean variables, which are inspected in Journey 5 but are not silently enabled here until their plugin/version-specific vertical contract is verified. Wave coupling is likewise demonstrated as a source-selection concept below; a point-spectrum fixture is not claimed to be a generated WWM boundary file.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from scripts.schism_case_data import assert_netcdf_contract, describe_fixture

for filename, variables in [("era5.nc", ("u10", "v10", "msl")), ("hycom.nc", ("surf_el", "water_u", "water_v", "temperature", "salinity"))]:
    assert_netcdf_contract(case / filename, variables=variables, dimensions=("time",))
    print(describe_fixture(case / filename))


In [ ]:
# A wave source is available for inspection, but it is point spectra rather than a
# SCHISM-WWM grid/catalog fixture, so this cell deliberately stops before generation.
wave_source = Path(root) / "tests" / "data" / "aus-20230101.nc"
if wave_source.is_file():
    waves = xr.open_dataset(wave_source)
    print("Wave source dimensions:", dict(waves.sizes))
    print("Wave variables:", list(waves.data_vars))
    waves.efth.isel(time=0, site=0).plot(x="freq", yincrease=False)
    plt.title("Available point wave spectrum (not WWM boundary output)")
    plt.show()
    waves.close()
else:
    print("Wave fixture unavailable; WWM generation remains an explicit follow-up.")


In [ ]:
component_files = {
    "grid": ["hgrid.gr3", "vgrid.in"],
    "atmos": ["sflux/*"],
    "tides": ["bctides.in"],
    "ocean elevation": ["elev2D.th.nc"],
    "ocean 3-D (inspected, not enabled)": ["uv3D.th.nc", "TEM_3D.th.nc", "SAL_3D.th.nc"],
    "waves (source inspected, generation blocked)": ["wavedata.nc"],
}
for component, outputs in component_files.items():
    print(f"{component:40s} -> {', '.join(outputs)}")
print("Prepared inputs are not model results: SCHISM/MPI execution and scientific validation remain optional and external.")
